# RNN_순환신경망
* 언어, 시계열 데이터 분석에 주로 사용
* 길이가 길어지면 기울기 소실문제 발생
* LSTM, GRU등으로 문제를 보완

In [4]:
from tensorflow.keras.preprocessing.text import Tokenizer, text_to_word_sequence
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Embedding
from tensorflow.keras.utils import to_categorical
import pandas as pd
import numpy as np

# 텐서플로에서 텍스트 전처리하기
* 토큰화: 문장을 단어 혹은 형태소로 쪼개는 것
* 원핫인코딩: 문자를 벡터화
* 임베딩(embedding): 벡터화 -> 원핫인코딩을 더 축소

In [5]:
text = "해보지 않으면 해낼 수 없다"

In [6]:
result = text_to_word_sequence(text)
result

['해보지', '않으면', '해낼', '수', '없다']

단어 빈도수 세기

In [14]:
docs = ['먼저 텍스트의 각 단어를 나우어 토큰화합니다',
       '텍스트의 단어를 토큰화해야 딥러닝에서 인식됩니다',
       '토큰화한 결과는 딥러닝에서 사용할 수 있습니다',
       '텍스트 전처리에는 벡터화 원핫인코딩 패딩으로 길이 맞추기 등이 필요합니다',
       '딥러닝 쉽지 않네요']

In [8]:
docs[2]

'토큰화한 결과는 딥러닝에서 사용할 수 있습니다'

In [15]:
token = Tokenizer()
token.fit_on_texts(docs)
print("단어 카운트:\n ", token.word_counts)
print("문장 카운트:\n ", token.document_count)
print("각 단어가 몇 개의 문장에 포함되어 있는지 계산:\n ", token.word_docs)
print("각 단어에 매겨진 인덱스 값:\n ", token.word_index)

단어 카운트:
  OrderedDict([('먼저', 1), ('텍스트의', 2), ('각', 1), ('단어를', 2), ('나우어', 1), ('토큰화합니다', 1), ('토큰화해야', 1), ('딥러닝에서', 2), ('인식됩니다', 1), ('토큰화한', 1), ('결과는', 1), ('사용할', 1), ('수', 1), ('있습니다', 1), ('텍스트', 1), ('전처리에는', 1), ('벡터화', 1), ('원핫인코딩', 1), ('패딩으로', 1), ('길이', 1), ('맞추기', 1), ('등이', 1), ('필요합니다', 1), ('딥러닝', 1), ('쉽지', 1), ('않네요', 1)])
문장 카운트:
  5
각 단어가 몇 개의 문장에 포함되어 있는지 계산:
  defaultdict(<class 'int'>, {'토큰화합니다': 1, '먼저': 1, '나우어': 1, '단어를': 2, '각': 1, '텍스트의': 2, '토큰화해야': 1, '인식됩니다': 1, '딥러닝에서': 2, '수': 1, '사용할': 1, '결과는': 1, '있습니다': 1, '토큰화한': 1, '벡터화': 1, '패딩으로': 1, '필요합니다': 1, '길이': 1, '등이': 1, '전처리에는': 1, '원핫인코딩': 1, '텍스트': 1, '맞추기': 1, '딥러닝': 1, '않네요': 1, '쉽지': 1})
각 단어에 매겨진 인덱스 값:
  {'텍스트의': 1, '단어를': 2, '딥러닝에서': 3, '먼저': 4, '각': 5, '나우어': 6, '토큰화합니다': 7, '토큰화해야': 8, '인식됩니다': 9, '토큰화한': 10, '결과는': 11, '사용할': 12, '수': 13, '있습니다': 14, '텍스트': 15, '전처리에는': 16, '벡터화': 17, '원핫인코딩': 18, '패딩으로': 19, '길이': 20, '맞추기': 21, '등이': 22, '필요합니다': 23, '딥러닝': 24, '쉽지': 25, '않네요': 26}


In [ ]:
텍스트의 단어로 토큰화해야 딥러닝에서 인식됩니다

In [ ]:
1 8 9 2 10

In [16]:
x = token.texts_to_sequences(docs)
x

[[4, 1, 5, 2, 6, 7],
 [1, 2, 8, 3, 9],
 [10, 11, 3, 12, 13, 14],
 [15, 16, 17, 18, 19, 20, 21, 22, 23],
 [24, 25, 26]]

In [18]:
lenofsent = []
for i in x:
    lenofsent.append(len(i))
max(lenofsent)

9

In [19]:
# 문장의 길이를 맞추기 위한 패딩
# 가장 긴 문장 길이 + 1의 길이로 패딩
# 문장 시작에는 0이 있어야 함
padded_x = pad_sequences(x, max([len(i) for i in x])+1)
padded_x

array([[ 0,  0,  0,  0,  4,  1,  5,  2,  6,  7],
       [ 0,  0,  0,  0,  0,  1,  2,  8,  3,  9],
       [ 0,  0,  0,  0, 10, 11,  3, 12, 13, 14],
       [ 0, 15, 16, 17, 18, 19, 20, 21, 22, 23],
       [ 0,  0,  0,  0,  0,  0,  0, 24, 25, 26]], dtype=int32)

# 텍스트를 읽고 긍정, 부정 예측하기

In [20]:
docs2 = ["너무 재밌네요",
        "최고예요",
        "참 신기한 딥러닝이네요",
        "인공지능 칭찬합니다",
        "더 자세히 배우고 싶어요",
        "변화가 너무 빨라요",
        "GPT성능이 생각보다 별로네요",
        "제미나이보다는 낫죠",
        "나는 차라리 라마를 쓴다",
        "유료 결재 싫어요"]

In [21]:
classes = np.array([1, 1, 1, 1, 1, 0, 0, 1, 0, 0])

In [22]:
token = Tokenizer()
token.fit_on_texts(docs2)
print(token.word_index)

{'너무': 1, '재밌네요': 2, '최고예요': 3, '참': 4, '신기한': 5, '딥러닝이네요': 6, '인공지능': 7, '칭찬합니다': 8, '더': 9, '자세히': 10, '배우고': 11, '싶어요': 12, '변화가': 13, '빨라요': 14, 'gpt성능이': 15, '생각보다': 16, '별로네요': 17, '제미나이보다는': 18, '낫죠': 19, '나는': 20, '차라리': 21, '라마를': 22, '쓴다': 23, '유료': 24, '결재': 25, '싫어요': 26}


In [23]:
x = token.texts_to_sequences(docs2)
print("토큰화 결과:\n", x)

토큰화 결과:
 [[1, 2], [3], [4, 5, 6], [7, 8], [9, 10, 11, 12], [13, 1, 14], [15, 16, 17], [18, 19], [20, 21, 22, 23], [24, 25, 26]]


In [26]:
padding_x = pad_sequences(x, max([len(i) for i in x]))
padding_x

array([[ 0,  0,  1,  2],
       [ 0,  0,  0,  3],
       [ 0,  4,  5,  6],
       [ 0,  0,  7,  8],
       [ 9, 10, 11, 12],
       [ 0, 13,  1, 14],
       [ 0, 15, 16, 17],
       [ 0,  0, 18, 19],
       [20, 21, 22, 23],
       [ 0, 24, 25, 26]], dtype=int32)

# 임베딩

In [27]:
word_size = len(token.word_index) + 1
word_size

27

In [ ]:
model = Sequential()
model.add(Embedding(word_size, 8, input_length=4))
model.add

In [ ]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(padding_x, classes, epochs=20)
print(model.evaluate(padding_x, classes)[1])